In [59]:
from datasets import DatasetDict, Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate
import numpy as np
import torch

In [2]:
ds = load_dataset("atulgupta002/banking_customer_service_query_intent",split='train')

In [3]:
print(ds)

Dataset({
    features: ['Unnamed: 0', 'query', 'intent'],
    num_rows: 5000
})


In [4]:
print(set(ds['intent']))

{'transaction_query', 'credi_card_application', 'loan_inquiry', 'password_reset', 'balance_inquiry', 'fraud_report'}


In [55]:
labels = [
    'transaction_query',
    'password_reset',
    'loan_inquiry',
    'fraud_report',
    'credi_card_application',
    'balance_inquiry'
]

label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

In [6]:
# Using a simple BERT model from google - lightweight (110 million parameters) but good for our task
model_name = "google-bert/bert-base-uncased"

# Impporting tokenizer associated with our model. This will tokenize our input text into vector embeddings that the model expects.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Attaching a 6-class classification head to the model. This means the model's initial weights will be randomly initialized.
## In other words, we need to train it for our task.
model = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=6,id2label=id2label,label2id=label2id)

/opt/anaconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### I am choosing not to freeze base model parameters.

In [8]:
# Let's encode the input

def encode(example):
    example['label'] = label2id[example['intent']]
    return example

ds = ds.map(encode)

In [9]:
def tokenize(example):
    return tokenizer(example['query'],truncation=True,padding='max_length')

tokenized_ds = ds.map(tokenize,batched=True)

In [10]:
tokenized_ds = tokenized_ds.remove_columns(["query", "intent","Unnamed: 0"])
# tokenized_ds.set_format("torch")

In [11]:
print(tokenized_ds)

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5000
})


In [12]:
# Split dataset into training (80%) and test (20%)
train_test_split = tokenized_ds.train_test_split(test_size=0.2, seed=42)

# Split the 20% test portion further into 10% validation and 10% test
train_dataset = train_test_split['train']
val_test_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)
val_dataset = val_test_split['train']
test_dataset = val_test_split['test']

In [13]:
train_dataset

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4000
})

In [14]:
val_dataset

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})

In [15]:
test_dataset

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})

In [16]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
)

trainer.train()


Epoch,Training Loss,Validation Loss
1,No log,0.121110
2,0.295200,0.082614
3,0.295200,0.063481


TrainOutput(global_step=750, training_loss=0.20113016160329183, metrics={'train_runtime': 65352.0999, 'train_samples_per_second': 0.184, 'train_steps_per_second': 0.011, 'total_flos': 3157446057984000.0, 'train_loss': 0.20113016160329183, 'epoch': 3.0})

In [37]:
test_results = trainer.evaluate(eval_dataset=test_dataset)
print("Test Results:", test_results)

Test Results: {'eval_loss': 0.05850950628519058, 'eval_runtime': 6590.5089, 'eval_samples_per_second': 0.076, 'eval_steps_per_second': 0.005, 'epoch': 3.0}


In [69]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class_id = logits.argmax().item()

    return id2label[predicted_class_id]

In [71]:
text = "how much do i have?"
print(predict(text))

balance_inquiry


In [75]:
text = "Where is my money??"
print(predict(text))

fraud_report


In [77]:
text = "change my pw"
print(predict(text))

password_reset


In [79]:
text = "can you give me money for short term"
print(predict(text))

loan_inquiry
